In [1]:
import os
from nwtrace import *
import pandas as pd
import geopandas as gpd

from pathlib import Path

In [2]:
lines = Path('data/more/full_sewers.geojson')
nodes = Path('data/more/all_node_connections.geojson')

lines_gdf = gpd.read_file(lines)
nodes_gdf = gpd.read_file(nodes)

In [3]:
multiple = True
upstream_only = False
downstream_only = False
verbose = True

sewer_id_field = 'FACILITYID'
upstream_field = 'FROMMH'
downstream_field = 'TOMH'

outfall_file = 'data/more/BC_outfalls.csv'
id_field = 'Asset Identification'

outfalls = pd.read_csv(outfall_file)[id_field].tolist()

target_endpoints = outfalls

outputname_extra = "allBC_"
output_dir = f"./out"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

result = []

sewershed = NWTrace(
    network=lines_gdf,
    id_field=sewer_id_field,
    upstream_field = upstream_field,
    downstream_field = downstream_field,
    verbose=verbose,
    output_dir=output_dir,
)

# additional connections
fittings = gpd.read_file("data/more/fitting_connections.geojson")
nodes_up = (fittings[["FACILITYID", "TO_FIXED"]]
            .dropna(subset=["FACILITYID", "TO_FIXED"]) # remove rows with None values
            .rename(columns={"FACILITYID": 'node_id', "TO_FIXED": 'segment_id'})
            .to_dict(orient="records"))

sewershed.add_upstream_nodes(nodes_up)


node_lookup, seg_lookup = sewershed.get_lookup_tables()


Added 5245 node-segment connection(s)
Created 627 new node(s)
Created 374 new segment(s).



In [10]:
errors = utils.verify_network_geometry(
    lines=lines_gdf,
    points=nodes_gdf,
    lookup_table=seg_lookup,
    line_id_field="FACILITYID",
    point_id_field="FACILITYID",
    threshold=100
)

100%|██████████| 169323/169323 [00:01<00:00, 103710.16it/s]

1176 Errors Found
Average Distance 0.14510461119443013 units


In [5]:
err_df = gpd.GeoDataFrame.from_dict(errors, geometry="geometry")
err_df = err_df.set_crs(lines_gdf.crs)

err_df.to_file("out/errors.gpkg", driver="GPKG", layer="errorsv1")

err_df

,node_id,segment_id,error_t,error_msg,dist,geometry
0,MH5212822131,SL0000-001-5481,missing node,node MH5212822131 does not exist in dataset,-1.000000,"MULTILINESTRING ((322146.483 4852349.718, 3221..."
1,MH4856521416,SL0000-012-0241,spatial,node MH4856521416 is more than 10 units from s...,53.931959,"MULTILINESTRING ((321483.472 4848889.549, 3214..."
2,MH5151728331,SL0000-017-3071,missing node,node MH5151728331 does not exist in dataset,-1.000000,"MULTILINESTRING ((328347.352 4851739.48, 32833..."
3,MH5194926776,SL0000-027-4231,missing node,node MH5194926776 does not exist in dataset,-1.000000,"MULTILINESTRING ((326791.821 4852171.698, 3267..."
4,MH5008926783,SL0000-027-9001,missing node,node MH5008926783 does not exist in dataset,-1.000000,"MULTILINESTRING ((326799.977 4850308.313, 3267..."
...,...,...,...,...,...,...
1220,None,SL4000-007,missing segment,segment SL4000-007 does not exist in dataset,-1.000000,None
1221,None,SL110491,missing segment,segment SL110491 does not exist in dataset,-1.000000,None
1222,None,SL1404719,missing segment,segment SL1404719 does not exist in dataset,-1.000000,None
1223,None,SL2000-011,missing segment,segment SL2000-011 does not exist in dataset,-1.000000,None


In [6]:
cons_err_df = err_df.groupby(["error_t"]).count()
cons_err_df

,node_id,segment_id,error_msg,dist,geometry
error_t,,,,,
missing node,789,789,789,789,789
missing segment,0,374,374,374,0
spatial,62,62,62,62,62


In [9]:
ids_dup = utils.count_duplicates(lines_gdf, "FACILITYID", minimum_count=0)
ids_dup

{'duplicate_count': {'SL0000-000-0091': 1,
  'SL0000-000-0181': 1,
  'SL0000-000-0271': 1,
  'SL0000-000-0361': 1,
  'SL0000-000-0451': 1,
  'SL0000-000-0541': 1,
  'SL0000-000-0631': 1,
  'SL0000-000-0721': 1,
  'SL0000-000-0722': 1,
  'SL0000-000-0811': 1,
  'SL0000-000-0901': 1,
  'SL0000-000-0991': 1,
  'SL0000-000-1081': 1,
  'SL0000-000-1171': 1,
  'SL0000-000-1261': 1,
  'SL0000-000-1351': 1,
  'SL0000-000-1441': 1,
  'SL0000-000-1531': 1,
  'SL0000-000-1621': 1,
  'SL0000-000-1711': 1,
  'SL0000-000-1801': 1,
  'SL0000-000-1891': 1,
  'SL0000-000-1981': 1,
  'SL0000-000-2071': 1,
  'SL0000-000-2161': 1,
  'SL0000-000-3961': 1,
  'SL0000-000-4051': 1,
  'SL0000-000-4141': 1,
  'SL0000-000-4231': 1,
  'SL0000-000-4232': 1,
  'SL0000-000-4321': 1,
  'SL0000-000-4411': 1,
  'SL0000-000-4501': 1,
  'SL0000-000-4591': 1,
  'SL0000-000-4681': 1,
  'SL0000-000-4771': 1,
  'SL0000-000-4861': 1,
  'SL0000-000-4951': 1,
  'SL0000-000-5041': 1,
  'SL0000-000-5131': 1,
  'SL0000-000-5221': 